# Quick Start 

In [ ]:
%load_ext autoreload
%autoreload 2

from fetta import HttpFetcher, BrowserManager, BrowserFetcher
from fetta_langchain import FettaLoader

In [ ]:
URLS = ["https://cnn.com/", "https://yahoo.com"]

## HTTP based

In [ ]:
# loader = FettaLoader(urls=URLS)
# docs = await loader.aload()
# docs

## Browser based

In [ ]:
try:
    # Init browser-based fetcher
    manager = BrowserManager()
    await manager.start()
    fetcher = BrowserFetcher(manager)
    
    # Fetch documents
    loader = FettaLoader(
        urls=URLS, 
        fetcher=fetcher,
        fetch_kwargs=dict(
            wait_until="domcontentloaded",
            timeout_ms=60_000
        ),
        include_sections=("text",)  # metadata, text, html
    )
    docs = await loader.aload()
    print(docs)
finally:
    await fetcher.close()
    await manager.stop()

## Summarize Documents

In [ ]:
from dotenv import load_dotenv

In [ ]:
from langchain_deepseek import ChatDeepSeek
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
# docs = splitter.split_documents(documents)

In [ ]:
def setup_llm():
    load_dotenv()
    return ChatDeepSeek(
        model="deepseek-flash", 
        temperature=0,
        max_tokens=200,
        extra_body={"thinking": {"type": "disabled"}},  # Disable thinking
        timeout=None,
        max_retries=2
    )

In [ ]:
# Initialize DeepSeek LLM
llm = setup_llm()
llm

In [ ]:
prompt = """
    Extract the 5-7 most important developments from the news below.
    
    For each:
    - One line, max 20 words.
    - Format: "<topic>: <what happened> - <why it matters>"
    - Rank by importance, not by the order they appear.
    
    News:
    {context}
"""

In [ ]:
# Create the summarization chain
prompt = ChatPromptTemplate.from_template(prompt)

# Build chain
chain = prompt | llm

# Run it asynchronously
result = await chain.ainvoke({"context": docs})
result

In [ ]:
print(result.content)

In [ ]:
# async for chunk in chain.astream({"context": docs}):
#     print(chunk, end="", flush=True)